#各月の気温・降水量・日照時間と、スーパー・百貨店での衣料品の売り上げの相関

In [2]:
'''
＜分析の流れ＞
欠損値の補完・削除
テストデータの分離
xとyへの分割
外れ値の除外
標準化（回帰）
訓練・検証データの分割

線形回帰・ラッソ回帰・リッジ回帰それぞれで
　訓練データで学習
　検証データで評価（K分割交差検証）

もっともよい回帰モデルにて、
　特徴量の絞り込み
　多項式特徴量の追加（回帰）
　交互作用特徴量の追加（回帰）

テストデータで最終評価

モデルのpickle保存（モデルを再利用したいとき）
'''

'\n＜分析の流れ＞\n欠損値の補完・削除\nテストデータの分離\nxとyへの分割\n外れ値の除外\n標準化（回帰）\n訓練・検証データの分割\n\n線形回帰・ラッソ回帰・リッジ回帰それぞれで\n\u3000訓練データで学習\n\u3000検証データで評価（K分割交差検証）\n\nもっともよい回帰モデルにて、\n\u3000特徴量の絞り込み\n\u3000多項式特徴量の追加（回帰）\n\u3000交互作用特徴量の追加（回帰）\n\nテストデータで最終評価\n\nモデルのpickle保存（モデルを再利用したいとき）\n'

In [3]:
#import

import pandas as pd

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [4]:
#データの読み込み
df_x = pd.read_csv('..\datafiles\crimate_data.csv',encoding="cp932")
df_y = pd.read_excel('..\datafiles\sales.xlsx')

display(df_x.head(5))
display(df_y.head(5))

,年月,日最高気温の平均(℃),日最高気温の平均(℃).1,日最高気温の平均(℃).2,日最低気温の平均(℃),日最低気温の平均(℃).1,日最低気温の平均(℃).2,降水量の合計(mm),降水量の合計(mm).1,降水量の合計(mm).2,...,日照時間(時間).3,平均風速(m/s),平均風速(m/s).1,平均風速(m/s).2,平均湿度(％),平均湿度(％).1,平均湿度(％).2,平均雲量(10分比),平均雲量(10分比).1,平均雲量(10分比).2
0,NaN,NaN,品質情報,均質番号,NaN,品質情報,均質番号,NaN,現象なし情報,品質情報,...,均質番号,NaN,品質情報,均質番号,NaN,品質情報,均質番号,NaN,品質情報,均質番号
1,5-Jan,10.0,8,1,2.6,8,1,77.0,0,8,...,1,3.7,8,1,47.0,8,1,4.2,8,1
2,5-Feb,9.9,8,1,2.5,8,1,48.0,0,8,...,1,3.9,8,1,45.0,8,1,6.3,8,1
3,5-Mar,13.1,8,1,5.0,8,1,71.0,0,8,...,1,3.5,8,1,49.0,8,1,6.0,8,1
4,5-Apr,19.6,8,1,10.7,8,1,81.0,0,8,...,1,3.8,8,1,54.0,8,1,5.4,8,1


,Unnamed: 0,2020年=100,2020年=100.1,2020年=100.2,2020年=100.3,2020年=100.4,2020年=100.5,2020年=100.6,2020年=100.7,2020年=100.8,2020年=100.9,2020年=100.10,2020年=100.11,Unnamed: 13,Unnamed: 14
0,NaN,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,C.Y.2020=100,NaN,NaN
1,NaN,合計,合計,合計,衣料品,衣料品,衣料品,飲食料品,飲食料品,飲食料品,その他,その他,その他,NaN,NaN
2,NaN,Total,Total,Total,Clothes,Clothes,Clothes,Food and Beverages,Food and Beverages,Food and Beverages,Others,Others,Others,NaN,NaN
3,NaN,合計,百貨店,スーパー,合計,百貨店,スーパー,合計,百貨店,スーパー,合計,百貨店,スーパー,NaN,NaN
4,年月,Total,Departmentstores,Supermarkets,Total,Departmentstores,Supermarkets,Total,Departmentstores,Supermarkets,Total,Departmentstores,Supermarkets,Month,Year


#目的変数データの整形

In [5]:
#目的変数データの異常値を取り除く

print(df_x.columns)

df_x.columns=['年月', '日最高気温の平均(℃)', '日最高気温の平均(℃).品質情報', '日最高気温の平均(℃).均質情報', 
    '日最低気温の平均(℃)','日最低気温の平均(℃).品質情報', '日最低気温の平均(℃).均質情報',
    '降水量の合計(mm)', '降水量の合計(mm).現象なし情報','降水量の合計(mm).品質情報', '降水量の合計(mm).均質情報',
    '日照時間(時間)', '日照時間(時間).現象なし情報', '日照時間(時間).品質情報','日照時間(時間).均質情報',
    '平均風速(m/s)', '平均風速(m/s).品質情報', '平均風速(m/s).均質情報',
    '平均湿度(％)','平均湿度(％).品質情報', '平均湿度(％).均質情報',
    '平均雲量(10分比)', '平均雲量(10分比).品質情報', '平均雲量(10分比).均質情報']

Index(['年月', '日最高気温の平均(℃)', '日最高気温の平均(℃).1', '日最高気温の平均(℃).2', '日最低気温の平均(℃)',
       '日最低気温の平均(℃).1', '日最低気温の平均(℃).2', '降水量の合計(mm)', '降水量の合計(mm).1',
       '降水量の合計(mm).2', '降水量の合計(mm).3', '日照時間(時間)', '日照時間(時間).1', '日照時間(時間).2',
       '日照時間(時間).3', '平均風速(m/s)', '平均風速(m/s).1', '平均風速(m/s).2', '平均湿度(％)',
       '平均湿度(％).1', '平均湿度(％).2', '平均雲量(10分比)', '平均雲量(10分比).1', '平均雲量(10分比).2'],
      dtype='object')


In [6]:
valid_check_cols=['日最高気温の平均(℃).品質情報','日最低気温の平均(℃).品質情報','降水量の合計(mm).品質情報','日照時間(時間).品質情報','平均風速(m/s).品質情報','平均湿度(％).品質情報','平均雲量(10分比).品質情報']

for col in valid_check_cols:
    print(df_x[col].value_counts())

日最高気温の平均(℃).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64
日最低気温の平均(℃).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64
降水量の合計(mm).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64
日照時間(時間).品質情報
8       248
5        10
品質情報      1
Name: count, dtype: int64
平均風速(m/s).品質情報
8       251
5         7
品質情報      1
Name: count, dtype: int64
平均湿度(％).品質情報
8       254
5         4
品質情報      1
Name: count, dtype: int64
平均雲量(10分比).品質情報
8       257
品質情報      1
5         1
Name: count, dtype: int64


In [7]:
#目的変数は正常値、準正常値のみであるから、除外すべき行はなし
#df_x2をデータ分析用のデータフレームとする

df_x2=df_x.copy()
to_drop_x=['日最高気温の平均(℃).品質情報', '日最高気温の平均(℃).均質情報', '日最低気温の平均(℃).品質情報', '日最低気温の平均(℃).均質情報',
    '降水量の合計(mm).現象なし情報','降水量の合計(mm).品質情報', '降水量の合計(mm).均質情報',
    '日照時間(時間).現象なし情報', '日照時間(時間).品質情報','日照時間(時間).均質情報',
    '平均風速(m/s).品質情報', '平均風速(m/s).均質情報',
    '平均湿度(％).品質情報', '平均湿度(％).均質情報',
    '平均雲量(10分比).品質情報', '平均雲量(10分比).均質情報']

df_x2=df_x2.drop(columns=to_drop_x,axis=1)
df_x2=df_x2.drop(index=0,axis=0)

#説明変数データの前処理

In [8]:
#説明変数の不要な行・列の削除
#df_y2をデータ分析用のデータフレームとする

df_y.columns=[
    '年月',
    '合計.全体','百貨店.全体','スーパー.全体',
    '合計.衣料品','百貨店.衣料品','スーパー.衣料品',
    '合計.飲食料品','百貨店.飲食料品','スーパー.飲食料品',
    '合計.その他','百貨店.その他','スーパー.その他',
    'Month','Year'
]


df_y2=df_y.copy()

to_drop_y=['百貨店.全体','スーパー.全体',
    '合計.衣料品','百貨店.衣料品','スーパー.衣料品',
    '合計.飲食料品','百貨店.飲食料品','スーパー.飲食料品',
    '合計.その他','百貨店.その他','スーパー.その他',
    'Month','Year'
]
df_y2=df_y2.drop(to_drop_y,axis=1)
df_y2=df_y2.drop(index=range(0,5),axis=0)

display(df_y2.head(5))
display(df_y2.tail(5))


,年月,合計.全体
5,2005年1月,123.7
6,2005年2月,97.3
7,2005年3月,111.9
8,2005年4月,110
9,2005年5月,110


,年月,合計.全体
256,2025年12月,144.8
257,2026年1月,120
258,2026年2月,107.3
259,2026年3月,119.3
260,2026年4月,112.2


#目的変数・説明変数双方のデータの前処理

In [9]:
print(df_x2.shape)
print(df_y2.shape)

(258, 8)
(256, 2)


In [10]:
df_x2=df_x2.drop(index=[257,258],axis=0)
print(df_x2.shape)

(256, 8)


In [11]:
#不要な行列の削除、欠損値の確認
df_y2=df_y2.drop(columns=['年月'],axis=1)
display(df_x2.isnull().sum())
display(df_y2.isnull().sum())

年月             0
日最高気温の平均(℃)    0
日最低気温の平均(℃)    0
降水量の合計(mm)     0
日照時間(時間)       0
平均風速(m/s)      0
平均湿度(％)        0
平均雲量(10分比)     0
dtype: int64

合計.全体    0
dtype: int64

In [12]:
#データ分析用の、k,tを連結したデータフレーム df3 の作成
df_x2 = df_x2.reset_index(drop=True)
df_y2 = df_y2.reset_index(drop=True)

df3 = pd.concat([df_x2, df_y2], axis=1)
df3=pd.concat([df_x2,df_y2],axis=1)
df3.index=df3[['年月']]
df3=df3.drop(columns=['年月'],axis=1)
display(df3.head(10))

,日最高気温の平均(℃),日最低気温の平均(℃),降水量の合計(mm),日照時間(時間),平均風速(m/s),平均湿度(％),平均雲量(10分比),合計.全体
"(5-Jan,)",10.0,2.6,77.0,200.0,3.7,47.0,4.2,123.7
"(5-Feb,)",9.9,2.5,48.0,148.9,3.9,45.0,6.3,97.3
"(5-Mar,)",13.1,5.0,71.0,175.1,3.5,49.0,6.0,111.9
"(5-Apr,)",19.6,10.7,81.0,216.1,3.8,54.0,5.4,110
"(5-May,)",21.9,14.1,180.5,172.3,3.9,58.0,7.2,110
"(5-Jun,)",26.7,20.2,170.5,119.3,3.0,70.0,8.9,109.9
"(5-Jul,)",29.1,22.6,247.5,103.9,2.9,71.0,8.9,123.6
"(5-Aug,)",31.8,25.1,189.5,159.9,3.4,68.0,7.6,104.9
"(5-Sep,)",28.2,21.8,177.5,154.2,3.6,67.0,7.1,101.6
"(5-Oct,)",22.3,16.6,201.5,108.3,3.3,69.0,7.3,112.2


In [13]:
#重回帰分析の目的変数、説明変数の定義
x=df3.loc[:,'日最高気温の平均(℃)':'平均雲量(10分比)']
y=df3[['合計.全体']]

In [14]:
display(df3.corr())

,日最高気温の平均(℃),日最低気温の平均(℃),降水量の合計(mm),日照時間(時間),平均風速(m/s),平均湿度(％),平均雲量(10分比),合計.全体
日最高気温の平均(℃),1.000000,0.990228,0.440030,-0.168859,0.108928,0.821434,0.728640,-0.245506
日最低気温の平均(℃),0.990228,1.000000,0.461464,-0.251917,0.098064,0.810628,0.755728,-0.226888
降水量の合計(mm),0.440030,0.461464,1.000000,-0.454171,0.042315,0.545993,0.572837,-0.273835
日照時間(時間),-0.168859,-0.251917,-0.454171,1.000000,0.228220,-0.446047,-0.644350,0.129160
平均風速(m/s),0.108928,0.098064,0.042315,0.228220,1.000000,-0.188491,0.092594,-0.205316
平均湿度(％),0.821434,0.810628,0.545993,-0.446047,-0.188491,1.000000,0.783452,-0.236799
平均雲量(10分比),0.728640,0.755728,0.572837,-0.644350,0.092594,0.783452,1.000000,-0.357664
合計.全体,-0.245506,-0.226888,-0.273835,0.129160,-0.205316,-0.236799,-0.357664,1.000000


In [15]:
#テストデータの分割
train_val,test=train_test_split(df3,test_size=0.2,random_state=0)

In [16]:
#x,y列の指定
x_col=[c for c in train_val.columns if c!='合計.全体']
y_col=['合計.全体']

#標準化
sc_model_x=StandardScaler().set_output(transform="pandas")
sc_model_x.fit(train_val[x_col])
sc_x=sc_model_x.transform(train_val[x_col])
sc_x_test=sc_model_x.transform(test[x_col])

In [17]:
#追加のimport
from sklearn.model_selection import KFold
kf=KFold(n_splits=3,shuffle=True,random_state=0)

from sklearn.model_selection import cross_validate
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso

In [18]:
#データの学習、評価（線形回帰）
model1=LinearRegression()
result=cross_validate(model1,sc_x,train_val[y_col],cv=kf,scoring='r2',return_train_score=True)
display(pd.DataFrame(result))
print(result['test_score'].mean())

,fit_time,score_time,test_score,train_score
0,0.012773,0.003414,0.177706,0.239061
1,0.002132,0.001519,0.180605,0.237655
2,0.002074,0.001646,0.167680,0.238880


0.17533046874994507


In [19]:
#データの学習、評価（リッジ回帰）
model2=Ridge(alpha=10)
result=cross_validate(model2,sc_x,train_val[y_col],cv=kf,scoring='r2',return_train_score=True)
display(pd.DataFrame(result))
print(result['test_score'].mean())

,fit_time,score_time,test_score,train_score
0,0.006998,0.005672,0.173493,0.211391
1,0.002232,0.001540,0.122216,0.227154
2,0.001976,0.001481,0.210685,0.180426


0.16879781216258458


In [20]:
#データの学習、評価（ラッソ回帰）
model3=Lasso(alpha=0.1)
result=cross_validate(model3,sc_x,train_val[y_col],cv=kf,scoring='r2',return_train_score=True)
display(pd.DataFrame(result))
print(result['test_score'].mean())

,fit_time,score_time,test_score,train_score
0,0.004527,0.001984,0.158429,0.219830
1,0.002331,0.001483,0.131193,0.230133
2,0.002277,0.002555,0.212444,0.204656


0.16735511062679953


In [21]:
#test_scoreの平均値が最も高い線形回帰にてモデルを作成する。

In [22]:
#test_scoreがいずれのモデルにおいても低い。多重共線性が疑われるため、VIF値を確認する。
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_df=pd.DataFrame()
vif_df["VIF_Factor"]=[variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=x_col
display(vif_df)

,VIF_Factor,features
0,158.894361,日最高気温の平均(℃)
1,128.098031,日最低気温の平均(℃)
2,1.688422,降水量の合計(mm)
3,5.786513,日照時間(時間)
4,1.799447,平均風速(m/s)
5,8.518083,平均湿度(％)
6,7.046147,平均雲量(10分比)


In [23]:
#最もVIF値が高かった['日最高気温の平均(℃)']を削除して結果を確認してみる
sc_x=sc_x.drop(columns=['日最高気温の平均(℃)'],axis=1)
result=cross_validate(model1,sc_x,train_val[y_col],cv=kf,scoring='r2',return_train_score=True)
display(pd.DataFrame(result))
print(result['test_score'].mean())

,fit_time,score_time,test_score,train_score
0,0.002701,0.001660,0.149205,0.221684
1,0.002490,0.001995,0.139242,0.232547
2,0.002231,0.001564,0.216541,0.187174


0.16832948811027723


In [24]:
#再度VIFの確認
vif_df=pd.DataFrame()
vif_df["VIF_Factor"]=[variance_inflation_factor(sc_x, i) for i in range(sc_x.shape[1])]
vif_df["features"]=sc_x.columns
display(vif_df)

,VIF_Factor,features
0,4.309258,日最低気温の平均(℃)
1,1.686934,降水量の合計(mm)
2,2.692449,日照時間(時間)
3,1.758036,平均風速(m/s)
4,5.261667,平均湿度(％)
5,6.743265,平均雲量(10分比)


In [25]:
#VIF値はいずれも10以下で問題ないと考えられるが、依然としてtest_scoreは低い。まだ精度改善の余地がある可能性がある。
#特徴量の絞り込みを行う。絞り込み前のテストスコア0.175と比較する。
for c in sc_x.columns:
    sc_x2=sc_x.copy()
    sc_x2=sc_x2.drop(columns=c,axis=1)
    result=cross_validate(model1,sc_x2,train_val[y_col],cv=kf,scoring='r2',return_train_score=True)
    print(f'列名＝{c}  テストスコアの平均＝{result['test_score'].mean()}')

列名＝日最低気温の平均(℃)  テストスコアの平均＝0.14958163011157902
列名＝降水量の合計(mm)  テストスコアの平均＝0.1494057402428557
列名＝日照時間(時間)  テストスコアの平均＝0.14781737004815734
列名＝平均風速(m/s)  テストスコアの平均＝0.16431207532772674
列名＝平均湿度(％)  テストスコアの平均＝0.17125506161156448
列名＝平均雲量(10分比)  テストスコアの平均＝0.13332016836111868


In [26]:
#改善が見られなかったため、特徴量の絞り込みはしない。
#多項式特徴量の導入を行う。

#指数を1~5まで変えて、自動的に実験。内側のループが完了後、多項式特徴量を導入して次のループへ
for c in sc_x.columns:
    best_exponent=0
    best_score=0
    for i in range(1,6):
        sc_x3=sc_x.copy()
        sc_x3[c]=sc_x3[c]**i
        sc_x3=sc_x3.drop(columns=c,axis=1)
        result=cross_validate(model1,sc_x3,train_val[y_col],cv=kf,scoring='r2',return_train_score=True)
        score=result['test_score'].mean()

        if best_score < score:
            best_exponent=i
            best_score=score
    
    print(f'導入した列＝{c}  最適指数＝{best_exponent}  導入した場合のスコア＝{best_score}')

導入した列＝日最低気温の平均(℃)  最適指数＝1  導入した場合のスコア＝0.14958163011157902
導入した列＝降水量の合計(mm)  最適指数＝1  導入した場合のスコア＝0.1494057402428557
導入した列＝日照時間(時間)  最適指数＝1  導入した場合のスコア＝0.14781737004815734
導入した列＝平均風速(m/s)  最適指数＝1  導入した場合のスコア＝0.16431207532772674
導入した列＝平均湿度(％)  最適指数＝1  導入した場合のスコア＝0.17125506161156448
導入した列＝平均雲量(10分比)  最適指数＝1  導入した場合のスコア＝0.13332016836111868


In [ ]:
#改善が見られなかったため、多項式特徴量の導入はしない。
#交互作用特徴量の導入。0.175以上のスコアが出ればその特徴量を表示
x_cols=sc_x.columns  #導入する交互作用特徴量は２次とする
for c1 in x_cols:
    best_exponent=0
    best_score=0
    for c2 in x_cols:
        if c1==c2:
            continue
        else:
            col_name=c1+'×'+c2
            sc_x4=sc_x.copy()
            sc_x4[col_name]=sc_x4[c1]*sc_x4[c2]

            result=cross_validate(model1,sc_x4,train_val[y_col],cv=kf,scoring='r2',return_train_score=True)
            score=result['test_score'].mean()

            if 0.175 <= score:
                print(f'列名＝{col_name}　テストスコア＝{score}')

列名＝日最低気温の平均(℃)×平均雲量(10分比)　テストスコア＝0.17701683009495597
列名＝降水量の合計(mm)×平均雲量(10分比)　テストスコア＝0.17589296559714432
列名＝平均雲量(10分比)×日最低気温の平均(℃)　テストスコア＝0.17701683009495597
列名＝平均雲量(10分比)×降水量の合計(mm)　テストスコア＝0.17589296559714432


In [ ]:
#大きな改善が見られなかったため、交互作用特徴量の導入はしない。
final_model=LinearRegression()

#完成したモデルをテストデータで評価
#学習・検証データを合わせたデータフレームを、ここでの学習データとして使用する。

final_model.fit(sc_x,train_val[y_col])
sc_x_test2=sc_x_test[sc_x.columns]
y_test=test[y_col]
score=final_model.score(sc_x_test2,y_test)
print(f'テストデータでのスコアは{score}')

テストデータでのスコアは0.07883578286386672
